<a href="https://colab.research.google.com/github/Jasp3r16/thesis_generative_timber/blob/main/23_new_stock_dataset_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** m23_new_stock_dataset_generation  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-02-20
### Properties script
**Goal:** to generate a new timber stock dataset, that the algorithm can use to choose elements for the design   
**Inputs:**
*   required properties
*   parameters

**Outputs:**
*   csv file that acts as the dataset

## Mounting Google drive

In [11]:
import os
import sys
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Definieer je paden (Pas de naam 'Thesis_Project' aan naar jouw mapnaam)
BASE_PATH = '/content/drive/MyDrive/Thesis_TU/'
DATA_PATH = os.path.join(BASE_PATH, 'data_thesis/')
SCRIPT_PATH = os.path.join(BASE_PATH, 'notebooks_thesis/')
OUTPUT_PATH = os.path.join(BASE_PATH, 'research_exports/')

# 3. Voeg de Scripts map toe aan het systeem-pad
# Hierdoor kun je .py bestanden uit die map 'importen'
if SCRIPT_PATH not in sys.path:
    sys.path.append(SCRIPT_PATH)

print(f"✅ Drive mounted. Project directory: {BASE_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted. Project directory: /content/drive/MyDrive/Thesis_TU/


## Parameters and inputs

In [12]:
# ==========================================
# CEL 1: INPUT PARAMETERS VOOR NEW STOCK
# ==========================================

# 1. Gewenste grootte van de dataset
# Omdat we nu veel afmetingen hebben, zetten we deze flink hoger
NUM_ELEMENTS = 10000

# 2. Geometrische Parameters
# Standaard handelslengtes (in mm, stappen van 300mm)
STANDARD_LENGTHS = [2400, 2700, 3000, 3300, 3600, 3900, 4200, 4500, 4800, 5100, 5400]

# Alle standaard doorsnedes (D, B) in mm gebaseerd op je referentie tabel
# Opgeslagen als (Width, Thickness) tuples.
STANDARD_SECTIONS = [
    # D = 100
    (100, 38), (100, 50), (100, 63), (100, 75), (100, 100),
    # D = 150
    (150, 38), (150, 50), (150, 63), (150, 75), (150, 100), (150, 150),
    # D = 175
    (175, 38), (175, 50), (175, 63), (175, 75),
    # D = 200
    (200, 38), (200, 50), (200, 63), (200, 75), (200, 100), (200, 150), (200, 200),
    # D = 225
    (225, 38), (225, 50), (225, 63), (225, 75),
    # D = 250
    (250, 50), (250, 75), (250, 100), (250, 250),
    # D = 300
    (300, 50), (300, 75), (300, 100), (300, 150), (300, 300)
]

# 3. Mechanische Parameters (C24 Referentiewaarden)
# Dictionary met Gemiddelde (mean) en Standaardafwijking (std)
MECH_PROPS = {
    'E_modulus': {'mean': 11000.0, 'std': 350.0}, # N/mm2
    'f_mk':      {'mean': 24.0,    'std': 1.0},   # N/mm2
    'density':   {'mean': 420.0,   'std': 15.0}   # kg/m3
}

print("Parameters succesvol geladen!")

Parameters succesvol geladen! Ga naar de volgende cel om te genereren.


## Start script

In [13]:
# ==========================================
# CEL 2: DATASET GENERATIE EN EXPORT
# ==========================================
import pandas as pd
import numpy as np
import random

def generate_full_timber_stock():
    data = []

    print(f"Genereren van {NUM_ELEMENTS} nieuwe houtelementen. Even geduld...")

    for i in range(NUM_ELEMENTS):
        # 1. Geometrie willekeurig selecteren uit de variabelen
        length = random.choice(STANDARD_LENGTHS)
        width, thickness = random.choice(STANDARD_SECTIONS)

        # Voeg kleine productietoleranties toe (bijv. max 0.5mm afwijking door zagen/schaven)
        w_act = np.random.normal(width, 0.5)
        t_act = np.random.normal(thickness, 0.5)

        # 2. Mechanische eigenschappen stochastisch genereren via normale verdeling
        e_mod = np.random.normal(MECH_PROPS['E_modulus']['mean'], MECH_PROPS['E_modulus']['std'])
        f_mk = np.random.normal(MECH_PROPS['f_mk']['mean'], MECH_PROPS['f_mk']['std'])
        dens = np.random.normal(MECH_PROPS['density']['mean'], MECH_PROPS['density']['std'])

        # 3. Data toevoegen aan de lijst (conform jouw kolom-structuur)
        data.append({
            'Member_ID': f"NS_{i:05d}",      # Bijv: N_00123
            'State': 0,                     # 0 = Nieuw hout
            'Length_Actual': int(length),
            'Width': round(w_act, 1),
            'Thickness': round(t_act, 1),
            'E_modulus_eff': round(e_mod, 1),
            'f_mk': round(f_mk, 1),
            'Density': round(dens, 1)
        })

    # 4. Omzetten naar Pandas DataFrame en Exporteren
    df_stock = pd.DataFrame(data)

    filename = f'{NUM_ELEMENTS}_new_timber_stock.csv'
    # sep=';' om te voorkomen dat Pandas extra aanhalingstekens genereert
    df_stock.to_csv(DATA_PATH + filename, index=False, sep=';')

    print(f"\nSucces! Dataset opgeslagen als '{filename}'.")
    return df_stock

# Voer de functie uit
df_new_stock = generate_full_timber_stock()

# Laat een random sample van 10 elementen zien om te bewijzen dat alle dimensies door elkaar staan
print("\nPreview van 10 willekeurige elementen uit de gegenereerde stock:")
display(df_new_stock.sample(10)) # display() werkt mooi als tabel in Google Colab

Genereren van 10000 nieuwe houtelementen. Even geduld...

Succes! Dataset opgeslagen als '10000_new_timber_stock.csv'.

Preview van 10 willekeurige elementen uit de gegenereerde stock:


,Member_ID,State,Length_Actual,Width,Thickness,E_modulus_eff,f_mk,Density
5354,NS_05354,0,2700,99.6,99.6,10982.3,26.4,430.4
4213,NS_04213,0,4200,299.3,300.4,11023.5,23.6,416.0
5591,NS_05591,0,5100,200.3,150.6,10573.5,23.0,404.9
7072,NS_07072,0,3300,100.2,50.2,10687.0,22.8,405.8
4211,NS_04211,0,2700,300.0,99.7,10925.2,25.3,437.9
2564,NS_02564,0,5100,199.7,149.5,10944.8,26.0,410.6
9250,NS_09250,0,3300,299.8,149.7,10629.0,22.9,432.8
7205,NS_07205,0,4200,99.8,37.6,10246.3,24.6,430.2
640,NS_00640,0,4800,300.8,150.2,10798.8,23.8,447.3
8179,NS_08179,0,4500,201.0,149.5,10614.9,23.7,411.8
